# Sedona-Sail on Wherobots: SpatialBench Q3

This runtime ships **sedona-sail** — [Sail](https://github.com/lakehq/sail), a Rust-native Spark Connect server, with the [SedonaDB](https://sedona.apache.org/sedonadb/) spatial kernels registered as its `ST_*` functions — in place of a JVM Spark cluster. The whole engine runs inside this notebook's Python process on the driver: no executors are started, and the interpreter is the free-threaded CPython 3.14t build, so Python UDFs run in parallel without the GIL.

This notebook:

1. starts a Sail server in-process and connects a Spark Connect session to it;
2. registers the [SpatialBench](https://sedona.apache.org/spatialbench/) `trip` table straight from S3;
3. runs SpatialBench **Q3** — monthly trip statistics for pickups near Sedona, Arizona — with Spatial SQL.

## Set up a Sail session

Where the other Wherobots examples create a `SedonaContext`, here we start Sail ourselves and connect to it over Spark Connect. Sail reads its configuration from environment variables **when the server starts**, so they are set first. The session object is named `sedona`, as in the other examples, so the rest of the notebook reads the same:

- `AWS_SKIP_SIGNATURE=true` — the SpatialBench bucket is public, so S3 requests are not signed. This applies to every S3 read in the session; leave it unset in notebooks that read private buckets.
- `SAIL_EXECUTION__DEFAULT_PARALLELISM` — Sail runs a little faster oversubscribed at 2× the pod's usable CPUs.

In [ ]:
import os
import sys
import time

try:
    usable_cpus = len(os.sched_getaffinity(0))
except AttributeError:
    usable_cpus = os.cpu_count() or 4

os.environ["AWS_SKIP_SIGNATURE"] = "true"
os.environ.setdefault("AWS_REGION", "us-west-2")
os.environ["SAIL_RUNTIME__MEMORY_POOL__TYPE"] = "unbounded"
os.environ["SAIL_EXECUTION__DEFAULT_PARALLELISM"] = str(2 * usable_cpus)

import pysail
from pysail.spark import SparkConnectServer
from pyspark.sql.connect.session import SparkSession

server = SparkConnectServer(ip="127.0.0.1", port=0)
server.start(background=True)
host, port = server.listening_address

sedona = SparkSession.builder.remote(f"sc://{host}:{port}").create()
print(f"pysail {pysail.__version__} · Spark Connect API {sedona.version} · "
      f"parallelism {2 * usable_cpus} · GIL enabled: {sys._is_gil_enabled()}")

## Load the SpatialBench `trip` table

SpatialBench is Apache Sedona's spatial benchmark. Its data is plain Parquet: the `trip` table stores pickup and drop-off locations as WKB, which is why the query below wraps them in `ST_GeomFromWKB`. Scale factor 1 is quick; scale factor 10 (about 60 million trips) is the scale used for the published single-node benchmarks of this runtime.

In [ ]:
SF = 10
TRIP = f"s3://wherobots-benchmark-prod/SpatialBench_sf={SF}_format=parquet/trip/"

trip = sedona.read.parquet(TRIP)
trip.createOrReplaceTempView("trip")
trip.printSchema()

## Run SpatialBench Q3

For every month, count the trips that started within 0.045° (about 5 km) of a box around Sedona, Arizona, and average their distance, duration and fare. The spatial predicate is `ST_DWithin` between each pickup point and the box; everything else is ordinary SQL.

SpatialBench's synthetic distances and fares are small decimals (the reference answer for this query has averages around `1e-05`), so those two columns read as fractions; the trip counts and durations are the readable part.

In [ ]:
Q3 = """
SELECT DATE_TRUNC('month', t.t_pickuptime) AS pickup_month,
       COUNT(t.t_tripkey)                      AS total_trips,
       AVG(t.t_distance)                       AS avg_distance,
       AVG(t.t_dropofftime - t.t_pickuptime)   AS avg_duration,
       AVG(t.t_fare)                           AS avg_fare
FROM trip t
WHERE ST_DWithin(
        ST_GeomFromWKB(t.t_pickuploc),
        ST_GeomFromText('POLYGON((-111.9060 34.7347, -111.6160 34.7347, -111.6160 35.0047, -111.9060 35.0047, -111.9060 34.7347))'),
        0.045)
GROUP BY pickup_month
ORDER BY pickup_month
"""

t0 = time.time()
rows = sedona.sql(Q3).collect()
print(f"Q3 on sf={SF}: {len(rows)} months in {time.time() - t0:.1f} s\n")

print(f"{'month':<8} {'trips':>8} {'avg distance':>13} {'avg duration':>16} {'avg fare':>11}")
for r in rows:
    print(f"{r.pickup_month:%Y-%m}  {r.total_trips:>8,} {r.avg_distance:>13.6f} "
          f"{str(r.avg_duration):>16} {r.avg_fare:>11.6f}")

## Shut down

Stop the session and the server to release the pod's memory. Restarting the kernel does the same.

In [ ]:
sedona.stop()
server.stop()